# hai07 - uji kelulusan model (apakah model lain layak menggantikan pembaca 100M?)
Input (Add Input): dataset **hai07-acceptance** (unggah `kaggle/hai07-acceptance.zip`; dibuat oleh `python tools/package_acceptance_kit.py` di repo, ±43 KB).
Settings: Accelerator = **GPU T4 x2** (atau P100), **Internet = ON** (mengunduh model dari HuggingFace), Persistence = Files.
Alur: jalankan dengan `QUICK = True` dulu (uji asap per model, beberapa menit) untuk memastikan model termuat dan tidak galat, lalu `QUICK = False` untuk uji penuh.
Hasil: tabel perbandingan dengan model sekarang di sel terakhir, dan laporan JSON per model di `/kaggle/working/reports`.

**Peringatan:**
- Daftar model di bawah adalah contoh dari ingatan; **belum diverifikasi** bahwa semuanya ada dan bisa diunduh saat Anda menjalankannya. Ganti sesuka Anda.
- **Periksa lisensi tiap model** sebelum memakainya di produk (beberapa berlisensi non-komersial atau perlu menerima syarat di HuggingFace dan memakai token).
- Backend HuggingFace di skrip ini baru diuji lewat model tiruan, **belum pernah dijalankan pada model sungguhan**. Itu sebabnya `QUICK = True` dulu.
- Kriteria lulus ada di `evalsets/acceptance_criteria.json` (disetujui pengembang 2026-11-22). Lulus berarti memenuhi SEMUA kriteria.

In [ ]:
QUICK = True          # True: cuplikan kecil per ujian (kriteria tidak berlaku); False: uji penuh
MODELS = [            # id HuggingFace atau folder model; contoh, belum diverifikasi, lisensi periksa sendiri
    "Qwen/Qwen2.5-0.5B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-3B-Instruct",
    "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    # "google/gemma-2-2b-it",               # perlu menerima syarat + token HuggingFace
    # "meta-llama/Llama-3.2-1B-Instruct",   # idem
]
HF_TOKEN_SECRET = "HF_TOKEN"   # nama rahasia Kaggle (Add-ons > Secrets) untuk model yang perlu token; boleh kosong
TIME_LIMIT_PER_MODEL_S = 40 * 60


In [ ]:
import glob, os, subprocess, sys, zipfile
WORK = "/kaggle/working/hai07-acceptance"
OUT = "/kaggle/working/reports"
os.makedirs(WORK, exist_ok=True); os.makedirs(OUT, exist_ok=True)
# paket bisa sudah terekstrak oleh Kaggle (folder dengan tools/model_acceptance_eval.py) atau masih berupa zip
src_dir = next((os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/tools/model_acceptance_eval.py", recursive=True)), None)
if src_dir:
    subprocess.run(["cp", "-r", src_dir + "/.", WORK], check=True)
else:
    zips = glob.glob("/kaggle/input/**/hai07-acceptance.zip", recursive=True)
    assert zips, "paket hai07-acceptance tidak ditemukan: Add Input -> dataset hai07-acceptance"
    zipfile.ZipFile(zips[0]).extractall(WORK)
os.chdir(WORK)
print("isi paket:", sorted(os.listdir(WORK)))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate"], check=True)
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret(HF_TOKEN_SECRET)
    if tok: os.environ["HF_TOKEN"] = tok; print("token HuggingFace terpasang dari rahasia Kaggle")
except Exception as e:
    print("tanpa token HuggingFace (", type(e).__name__, ") - model yang perlu persetujuan tidak bisa diunduh")
import torch, transformers
print("transformers", transformers.__version__, "| GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")


In [ ]:
import re, time
results = {}
for m in MODELS:
    safe = re.sub(r"[^A-Za-z0-9._-]+", "_", m)
    out = f"{OUT}/acceptance_{safe}{'_cepat' if QUICK else ''}.json"
    cmd = [sys.executable, "-u", "tools/model_acceptance_eval.py", "--backend", "hf", "--model", m, "--out", out] + (["--quick"] if QUICK else [])
    print("=" * 100); print("MODEL:", m); t0 = time.time()
    try:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        tail = []
        for line in p.stdout:
            print(line, end=""); tail.append(line)
            if time.time() - t0 > TIME_LIMIT_PER_MODEL_S:
                p.kill(); raise TimeoutError(f"melebihi {TIME_LIMIT_PER_MODEL_S // 60} menit")
        p.wait()
        results[m] = {"kode": p.returncode, "detik": round(time.time() - t0), "json": out if os.path.exists(out) else None, "galat": "".join(tail[-6:]) if p.returncode else ""}
    except Exception as e:
        results[m] = {"kode": -1, "detik": round(time.time() - t0), "json": None, "galat": f"{type(e).__name__}: {e}"}
    print(f"-> {m}: kode keluar {results[m]['kode']} dalam {results[m]['detik']} dtk")


In [ ]:
import json
# pembanding: model sekarang (gpt_sft_best.pt), dari reports/acceptance_gpt100m.json dan acceptance_gpt100m_final.json di repo
BASELINE = {"model": "gpt_sft_best.pt (sekarang)", "param": 100.9, "router": "18.1/16.7/15.4%", "bajak": 37, "tulisan": "11 benar/1 salah",
            "rumit": "29 benar/9 salah", "kasus": "20/26", "tim": "0/3", "tahan": "98.2%", "gagal": "8 dari 14"}
def row(m, j):
    r = j.get("router") or {}
    gagal = [k for k in j.get("kriteria", []) if not k["lolos"]]
    return {"model": m.split("/")[-1], "param": (j.get("info") or {}).get("parameter_juta"),
            "router": "/".join(f"{100 * v['akurasi']:.1f}" for v in r.values()) + "%" if r else "-",
            "bajak": sum(v["pembajakan_cari"] for v in r.values()) if r else "-",
            "tulisan": "{benar} benar/{salah} salah".format(**j["qa_beku"]["tulisan_tangan"]) if "qa_beku" in j else "-",
            "rumit": "{benar} benar/{salah} salah".format(**j["qa_beku"]["kalimat_rumit"]) if "qa_beku" in j else "-",
            "kasus": f"{sum(k['lolos'] for k in j['kasus'])}/{len(j['kasus'])}" if "kasus" in j else "-",
            "tim": (lambda t: f"{sum(k['lolos'] for k in t)}/{len(t)}")([k for k in j.get("kasus", []) if k["id"].startswith("rapat-tim")]) if "kasus" in j else "-",
            "tahan": f"{100 * j['ketahanan']['lolos_varian']:.1f}%" if "ketahanan" in j else "-",
            "gagal": "uji cepat: tidak berlaku" if j.get("cepat") else (f"{len(gagal)} dari {len(j['kriteria'])}" + ("  LULUS SEMUA" if not gagal else "")),
            "dtk/baca": j.get("detik_per_baca_median", "-")}
rows = [BASELINE]
for m, r in results.items():
    if r["json"]:
        rows.append(row(m, json.load(open(r["json"], encoding="utf-8"))))
    else:
        rows.append({"model": m.split("/")[-1], "gagal": "GALAT: " + r["galat"].strip().splitlines()[-1][:80] if r["galat"].strip() else "GALAT"})
cols = ["model", "param", "router", "bajak", "tulisan", "rumit", "kasus", "tim", "tahan", "dtk/baca", "gagal"]
w = {c: max(len(c), *(len(str(x.get(c, "-"))) for x in rows)) for c in cols}
print("  ".join(c.ljust(w[c]) for c in cols)); print("  ".join("-" * w[c] for c in cols))
for x in rows: print("  ".join(str(x.get(c, "-")).ljust(w[c]) for c in cols))
print("\nMode:", "UJI CEPAT (cuplikan kecil; kriteria tidak berlaku; pastikan semua model termuat tanpa galat, lalu jalankan QUICK=False)" if QUICK else "UJI PENUH (kriteria berlaku)")
print("Laporan JSON:", OUT)


## Cara membaca hasil
- **Lulus** = memenuhi **semua** kriteria di `acceptance_criteria.json`. Tabel menunjukkan jumlah kriteria yang gagal per model; model sekarang gagal 8 dari 14.
- **Kolom `bajak`** = pesan tentang catatan yang salah dialihkan ke alat (syarat 0). **`tim`** = tiga kasus nama tim yang gagal pada model sekarang.
- **Jangan menilai dari satu angka.** Model yang pintar memilih alat tetapi menjawab salah dari catatan tidak layak; keduanya dinilai.
- **Waktu per baca** hanya informasi di GPU Kaggle. Di komputer pemakai (CPU) model 1 sampai 3 miliar parameter jauh lebih lambat dan lebih besar dari 404 MB; ukur sendiri di komputer sasaran.
- Set uji kecil (puluhan soal per ujian) dan ditulis oleh pengembang. Lulus di sini belum menjamin kualitas di lapangan; itu syarat perlu, bukan syarat cukup.
- Bila ada model yang lulus: langkah berikutnya adalah memeriksa lisensinya, menguji kecepatan dan ukuran di komputer sasaran, lalu menyambungkannya sebagai pembaca di aplikasi (penjaga, sumber, dan koreksi tetap dipertahankan karena model bebas bisa mengarang).